# Couche sémantique : recherche en langage naturel sur un catalogue de données

Notebook **autonome** : tout le code vit ici, rien n'est importé depuis `src/` ou
`docmaker/`. L'objectif n'est pas la couverture de production (voir `src/retrieval/`
pour la version utilisée par le pipeline) mais la **compréhension et le bidouillage** :
chaque brique est une classe ou une fonction courte, commentée sur le *pourquoi*, et
les quelques variables qui comptent sont regroupées dans une seule cellule de
configuration (juste en dessous).

## Ce que fait ce notebook

Recherche en langage naturel sur un catalogue de tables/colonnes (identifiants,
descriptions, tags, glossaire) — sans Elasticsearch ni base vectorielle : des
embeddings locaux (CPU, quelques dizaines de Mo) et un produit scalaire numpy. À
l'échelle d'un datamart (10²–10³ tables), c'est exact et instantané ; un moteur dédié
ne se justifie qu'au-delà.

## Concepts couverts

1. **Déplier les identifiants** — `DMT_CPT_SLD_J` ne veut rien dire pour un modèle
   d'embedding généraliste ; « datamart compte solde journalier » si.
2. **Deux canaux d'embedding** — identité (le nom) et sens (la description) sont
   embeddés séparément, pas moyennés, pour qu'une description ne puisse jamais faire
   *baisser* le score d'une table déjà bien nommée.
3. **Facettes vs sémantique** — schéma, type d'entité, tags de classification servent
   de *filtres* (masque sur les scores), jamais embeddés : un tag dilue un vecteur
   sans jamais répondre à une question en langage naturel.
4. **L'index comme artefact** — persisté sur disque, versionnable, avec une empreinte
   du corpus pour détecter qu'il est périmé.
5. **Mesurer, pas regarder** — un petit « golden set » question → table attendue pour
   comparer deux configurations autrement qu'à l'œil.

## Deux sources de données (`DATA_SOURCE` ci-dessous)

- `"synthetic"` (défaut) : un mini-catalogue bancaire écrit à la main, **aucune
  infra requise** — le notebook tourne de bout en bout hors ligne.
- `"omd"` : le vrai catalogue [OpenMetadata](https://open-metadata.org/) d'un service
  connecté, via son SDK Python (GET uniquement). Nécessite une instance OMD et un
  jeton dans `.env` (voir `.env.example` à la racine du dépôt).

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import time
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("semantic_layer")

# ============================================================================
# CONFIGURATION -- tout ce qui est tweakable est ici. Le reste du notebook ne
# relit jamais une valeur en dur : pour une experience, on change une ligne ici.
# ============================================================================

DATA_SOURCE = "synthetic"  # "synthetic" (aucune infra) | "omd" (catalogue OpenMetadata reel)

# Modele d'embedding : n'importe quel modele supporte par fastembed. Multilingue
# et petit par defaut (bon compromis CPU) ; essayer par ex. "BAAI/bge-small-en-v1.5"
# (anglais, plus precis sur du texte anglais) pour comparer.
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_CACHE_DIR = Path(
    os.environ.get("DOCMAKER_MODEL_CACHE", Path.home() / ".cache" / "docmaker" / "embeddings")
)

# Poids du canal "sens" (description + glossaire) dans le score final. 0 = identite
# seule, 1 = sens seul. Voir section 5 pour la justification de la formule complete.
MEANING_WEIGHT = 0.5

INCLUDE_COLUMNS = True  # False = un document par table seulement (plus rapide, moins fin)
INDEX_DIR = Path("build") / "semantic_layer"  # artefact persiste : embeddings + documents

# Uniquement si DATA_SOURCE == "omd" :
OMD_SERVICE = "banking db"
OMD_SCHEMAS = {"ref", "stg", "ods", "dmt", "tec"}  # schemas metier, hors schemas systeme Oracle

## 1. Déplier les identifiants

Un dictionnaire d'abréviations maison, en attendant que ce vocabulaire vive dans un
glossaire gouverné (le paramètre `extra`, plus bas). `expand_identifier` consulte
d'abord les synonymes fournis en argument — le vocabulaire métier, quand il existe,
prime toujours sur ce repli codé en dur.

In [ ]:
ABBREVIATIONS = {
    # entites metier
    "CPT": "compte", "CLI": "client", "CRD": "credit", "MVT": "mouvement",
    "SLD": "solde", "OPE": "operation", "DOS": "dossier", "ADR": "adresse",
    "AGE": "agence", "ORG": "organisation", "PM": "personne morale",
    "PP": "personne physique",
    # mesures
    "MT": "montant", "NB": "nombre", "TX": "taux", "TXC": "taux de change",
    "TXI": "taux d'interet", "ENC": "encours", "EXP": "exposition", "TOT": "total",
    "RSQ": "risque",
    # temps
    "DT": "date", "JR": "jour", "J": "journalier", "M": "mensuel", "MOIS": "mois",
    "DEB": "debut", "FIN": "fin", "OUV": "ouverture",
    # techniques / referentiel
    "CD": "code", "ID": "identifiant", "LIB": "libelle", "TYP": "type", "NAT": "nature",
    "DEV": "devise", "CHF": "franc suisse", "PAY": "pays",
    "REF": "referentiel", "STG": "staging", "ODS": "donnees operationnelles",
    "DMT": "datamart", "DIM": "dimension", "F": "fait", "D": "dimension", "H": "historique",
    "BCK": "sauvegarde", "OLD": "ancien",
}


def expand_identifier(identifier: str, extra: dict[str, str] | None = None) -> str:
    """Deplie `DMT_CPT_SLD_J` en `"datamart compte solde journalier"`.

    `extra` (typiquement les synonymes d'un glossaire) est consulte en premier :
    le vocabulaire gouverne prime sur ce dictionnaire code en dur.
    """
    tokens = identifier.upper().replace(".", "_").split("_")
    lookup = {**ABBREVIATIONS, **(extra or {})}
    return " ".join(lookup.get(token, token.lower()) for token in tokens)


expand_identifier("DMT_CPT_SLD_J")

## 2. Modèle de données

Trois familles de structures, volontairement plates :

- `GlossaryEntry` — un terme métier : nom, définition, synonymes.
- `RawTable` / `RawColumn` — une vue normalisée du catalogue, **indépendante de la
  source** (synthétique ou OpenMetadata) : le reste du notebook ne connaît que ces
  deux classes, jamais les types OpenMetadata eux-mêmes. C'est le point de couture
  qui permet d'ajouter une troisième source plus tard sans toucher au reste.
- `Document` — l'unité indexable, avec le texte d'**identité** et le texte de
  **sens** séparés (raison détaillée en section 4), et des facettes structurées
  qui ne sont jamais embeddées.

In [ ]:
@dataclass
class GlossaryEntry:
    """Un terme de glossaire, reduit a ce qui sert au retrieval."""

    fqn: str
    name: str
    description: str = ""
    synonyms: list[str] = field(default_factory=list)

    def as_text(self) -> str:
        return " ".join(filter(None, [self.name, *self.synonyms, self.description]))


@dataclass
class RawColumn:
    """Une colonne, telle que la source (synthetique ou OMD) la fournit."""

    name: str
    data_type: str = ""
    description: str = ""
    glossary_terms: list[str] = field(default_factory=list)  # FQN de termes de glossaire
    tags: list[str] = field(default_factory=list)  # tags de classification (PII, ...)


@dataclass
class RawTable:
    fqn: str
    name: str
    schema: str
    description: str = ""
    glossary_terms: list[str] = field(default_factory=list)
    tags: list[str] = field(default_factory=list)
    columns: list[RawColumn] = field(default_factory=list)


@dataclass
class Catalog:
    """Instantane du catalogue : les tables et le glossaire qu'elles referencent."""

    tables: list[RawTable]
    glossary: dict[str, GlossaryEntry]  # indexe par FQN

    def synonym_map(self) -> dict[str, str]:
        """Synonymes du glossaire indexes par jeton majuscule (`SLD` -> "solde"),
        pour que le vocabulaire gouverne prime sur `ABBREVIATIONS`.
        """
        mapping: dict[str, str] = {}
        for entry in self.glossary.values():
            for synonym in entry.synonyms:
                token = synonym.strip().upper()
                if token and " " not in token:
                    mapping[token] = entry.name.lower()
        return mapping


@dataclass
class Document:
    """Unite indexable : une table ou une colonne."""

    fqn: str
    entity_type: str  # "table" | "column"
    name: str
    schema: str
    identity_text: str  # identifiants deplies
    meaning_text: str = ""  # description + glossaire ; vide tant que rien n'est documente
    tags: list[str] = field(default_factory=list)  # classification uniquement
    has_description: bool = False

    @property
    def has_meaning(self) -> bool:
        return bool(self.meaning_text.strip())

    def to_dict(self) -> dict:
        return {
            "fqn": self.fqn, "entity_type": self.entity_type, "name": self.name,
            "schema": self.schema, "identity_text": self.identity_text,
            "meaning_text": self.meaning_text, "tags": self.tags,
            "has_description": self.has_description,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "Document":
        return cls(**data)

## 3. Charger le catalogue

Deux implémentations, une seule interface (`Catalog`) : le reste du notebook ne sait
pas laquelle a été utilisée. `load_synthetic_catalog` fabrique un mini-catalogue
bancaire à la main, en **deux variantes** (`enriched=False/True`) pour rejouer
l'expérience de la section 8 (l'effet des métadonnées sur la pertinence).
`load_omd_catalog` interroge une vraie instance OpenMetadata (GET seulement, jamais
d'écriture — la boucle d'écriture réelle vit dans `src/sink/omd.py`).

`ODS_F_CPT_MVT_BCK_2019` est volontairement présente et jamais documentée : une
table de sauvegarde qui pollue le classement, pour illustrer la section 9.

In [ ]:
def load_synthetic_catalog(enriched: bool) -> Catalog:
    """`enriched=False` : identifiants seuls, catalogue "a l'etat reel" (peu ou pas
    documente). `enriched=True` : + descriptions + glossaire, pour mesurer l'ecart.
    """

    def col(name, data_type="VARCHAR2", description="", terms=None):
        return RawColumn(
            name=name, data_type=data_type,
            description=description if enriched else "",
            glossary_terms=(terms or []) if enriched else [],
        )

    tables = [
        RawTable(
            fqn="DMT.DMT_CPT_MVT_J", name="DMT_CPT_MVT_J", schema="dmt",
            description=(
                "Mouvements comptables journaliers par compte : une ligne par "
                "operation debitrice ou creditrice." if enriched else ""
            ),
            columns=[
                col("id_mvt", "NUMBER"),
                col("id_compte", "NUMBER"),
                col("dt_mvt", "DATE"),
                col("cd_typ_ope", "VARCHAR2",
                    "Code du type d'operation : debit, credit, virement, prelevement."),
                col("mt_mvt", "NUMBER", "Montant de l'operation dans la devise d'origine."),
            ],
        ),
        RawTable(
            fqn="DMT.DMT_CPT_SLD_J", name="DMT_CPT_SLD_J", schema="dmt",
            description=(
                "Solde de fin de journee par compte, en francs suisses." if enriched else ""
            ),
            columns=[col("id_compte", "NUMBER"), col("dt_jour", "DATE"), col("mt_sld_chf", "NUMBER")],
        ),
        RawTable(
            fqn="ODS.ODS_D_CLI_ADR", name="ODS_D_CLI_ADR", schema="ods",
            description=(
                "Adresses postales des clients, historisees par periode de validite."
                if enriched else ""
            ),
            columns=[
                col("id_client", "NUMBER"),
                col("rue", "VARCHAR2", "Libelle de voie de l'adresse postale."),
                col("ville", "VARCHAR2"),
                col("dt_deb_val", "DATE"),
            ],
        ),
        RawTable(
            fqn="DMT.DMT_F_CRD_ENC_M", name="DMT_F_CRD_ENC_M", schema="dmt",
            description="Encours de credit mensuels par dossier et par agence." if enriched else "",
            columns=[
                col("id_dossier", "NUMBER"),
                col("dt_fin_mois", "DATE"),
                col("mt_crd_restant", "NUMBER",
                    "Capital restant du sur le dossier de credit a la fin du mois.",
                    terms=["Banque.Encours"]),
            ],
        ),
        RawTable(
            fqn="REF.V_REF_FIN_TXC_CHF", name="V_REF_FIN_TXC_CHF", schema="ref",
            description=(
                "Cours de conversion quotidiens des devises vers le franc suisse."
                if enriched else ""
            ),
            columns=[col("devise", "VARCHAR2"), col("dt_jour", "DATE"), col("tx_chf", "NUMBER")],
        ),
        # Table de sauvegarde, jamais documentee : voir section 9 (pollution du classement).
        RawTable(
            fqn="ODS.ODS_F_CPT_MVT_BCK_2019", name="ODS_F_CPT_MVT_BCK_2019", schema="ods",
            columns=[col("id_mvt", "NUMBER"), col("mt_mvt", "NUMBER")],
        ),
    ]

    glossary: dict[str, GlossaryEntry] = {}
    if enriched:
        entries = [
            GlossaryEntry("Banque.Solde disponible", "Solde disponible",
                          "Montant utilisable immediatement par le client.",
                          ["SLD", "avoir disponible"]),
            GlossaryEntry("Banque.Mouvement", "Mouvement",
                          "Ecriture comptable passee sur un compte.",
                          ["MVT", "operation", "transaction"]),
            GlossaryEntry("Banque.Encours", "Encours",
                          "Capital restant du sur un credit a une date donnee.", ["ENC"]),
            GlossaryEntry("Banque.Taux de change", "Taux de change",
                          "Cours de conversion d'une devise vers une autre.", ["TXC", "TX"]),
        ]
        glossary = {e.fqn: e for e in entries}

    return Catalog(tables=tables, glossary=glossary)

In [ ]:
def load_omd_catalog(service: str, schemas: set[str]) -> Catalog:
    """GET uniquement, via le SDK `openmetadata-ingestion`. Necessite OMD_HOST_PORT /
    OMD_JWT_TOKEN dans `.env` (voir `.env.example`). Import differe : les cellules
    precedentes restent utilisables sans cette dependance installee.
    """
    from metadata.generated.schema.entity.data.glossaryTerm import GlossaryTerm
    from metadata.generated.schema.entity.data.table import Table as OMDTable
    from metadata.generated.schema.entity.services.connections.metadata.openMetadataConnection import (
        OpenMetadataConnection,
    )
    from metadata.generated.schema.security.client.openMetadataJWTClientConfig import (
        OpenMetadataJWTClientConfig,
    )
    from metadata.generated.schema.type.tagLabel import TagSource
    from metadata.ingestion.ometa.ometa_api import OpenMetadata

    connection = OpenMetadataConnection(
        hostPort=os.environ["OMD_HOST_PORT"],
        securityConfig=OpenMetadataJWTClientConfig(jwtToken=os.environ["OMD_JWT_TOKEN"]),
    )
    client = OpenMetadata(connection)

    glossary: dict[str, GlossaryEntry] = {}
    for term in client.list_all_entities(entity=GlossaryTerm, fields=["relatedTerms"]):
        fqn = term.fullyQualifiedName.root if term.fullyQualifiedName else term.name.root
        glossary[fqn] = GlossaryEntry(
            fqn=fqn, name=(term.displayName or term.name.root),
            description=term.description.root if term.description else "",
            synonyms=[s.root if hasattr(s, "root") else str(s) for s in (term.synonyms or [])],
        )

    def split_tags(tags) -> tuple[list[str], list[str]]:
        glossary_fqns, classification_fqns = [], []
        for label in tags or []:
            fqn = label.tagFQN.root if hasattr(label.tagFQN, "root") else str(label.tagFQN)
            (glossary_fqns if label.source == TagSource.Glossary else classification_fqns).append(fqn)
        return glossary_fqns, classification_fqns

    def adapt_column(c) -> RawColumn:
        glossary_fqns, classification_fqns = split_tags(c.tags)
        return RawColumn(
            name=c.name.root, data_type=c.dataTypeDisplay or "",
            description=c.description.root if c.description else "",
            glossary_terms=glossary_fqns, tags=classification_fqns,
        )

    tables: list[RawTable] = []
    for t in client.list_all_entities(
        entity=OMDTable, fields=["columns", "tags", "owners"], params={"service": service}
    ):
        schema = t.databaseSchema.name if t.databaseSchema else ""
        if schemas and schema.lower() not in schemas:
            continue
        glossary_fqns, classification_fqns = split_tags(t.tags)
        tables.append(RawTable(
            fqn=t.fullyQualifiedName.root, name=t.name.root, schema=schema,
            description=t.description.root if t.description else "",
            glossary_terms=glossary_fqns, tags=classification_fqns,
            columns=[adapt_column(c) for c in (t.columns or [])],
        ))

    log.info("catalogue OMD : %d table(s), %d terme(s) de glossaire", len(tables), len(glossary))
    return Catalog(tables=tables, glossary=glossary)

In [ ]:
def get_catalog(source: str = DATA_SOURCE, enriched: bool = True) -> Catalog:
    if source == "synthetic":
        return load_synthetic_catalog(enriched=enriched)
    if source == "omd":
        return load_omd_catalog(OMD_SERVICE, OMD_SCHEMAS)
    raise ValueError(f"DATA_SOURCE inconnu : {source!r}")


def stats(catalog: Catalog) -> dict:
    columns = [c for t in catalog.tables for c in t.columns]
    return {
        "tables": len(catalog.tables),
        "colonnes": len(columns),
        "tables decrites": sum(1 for t in catalog.tables if t.description),
        "colonnes decrites": sum(1 for c in columns if c.description),
        "termes de glossaire": len(catalog.glossary),
    }


catalog = get_catalog()
stats(catalog)

## 4. Composer les documents indexables

Chaque document porte deux textes embeddés séparément — l'**identité** (identifiants
dépliés) et le **sens** (description + glossaire) — et des facettes structurées
(schéma, type, tags) qui servent de filtres et ne sont jamais embeddées.

Deux vecteurs et pas un seul parce qu'un vecteur unique moyenne : ajouter une
description à une table déjà bien nommée ferait alors *baisser* son score. Le score
final est le `max` des deux canaux (section 5).

In [ ]:
def _glossary_text(fqns: list[str], glossary: dict[str, GlossaryEntry]) -> str:
    return " ; ".join(glossary[fqn].as_text() for fqn in fqns if fqn in glossary)


def _join(*parts: str) -> str:
    return "\n".join(p.strip() for p in parts if p and p.strip())


def build_documents(catalog: Catalog, include_columns: bool = INCLUDE_COLUMNS) -> list[Document]:
    """Un document par table (agrege les noms de colonnes : la granularite utile au
    text-to-SQL, "quelle table repond a cette question ?"), et par colonne si demande
    (les questions qui visent un champ precis).
    """
    synonyms = catalog.synonym_map()
    documents: list[Document] = []

    for table in catalog.tables:
        documents.append(Document(
            fqn=table.fqn, entity_type="table", name=table.name, schema=table.schema,
            identity_text=_join(
                f"table {expand_identifier(table.name, synonyms)}",
                "colonnes : " + ", ".join(expand_identifier(c.name, synonyms) for c in table.columns),
            ),
            meaning_text=_join(table.description, _glossary_text(table.glossary_terms, catalog.glossary)),
            tags=table.tags, has_description=bool(table.description),
        ))
        if not include_columns:
            continue
        for column in table.columns:
            documents.append(Document(
                fqn=f"{table.fqn}.{column.name}", entity_type="column", name=column.name,
                schema=table.schema,
                identity_text=_join(
                    f"colonne {expand_identifier(column.name, synonyms)} "
                    f"de la table {expand_identifier(table.name, synonyms)}",
                    f"type {column.data_type}",
                ),
                meaning_text=_join(column.description, _glossary_text(column.glossary_terms, catalog.glossary)),
                tags=column.tags, has_description=bool(column.description),
            ))
    return documents


documents = build_documents(catalog)
print(f"{len(documents)} documents\n")
exemple = next(d for d in documents if d.name == "DMT_CPT_MVT_J")
print("identite :", exemple.identity_text)
print("sens     :", exemple.meaning_text or "(rien de documente)")
print("facettes :", exemple.schema, exemple.entity_type, exemple.tags)

## 5. Indexer : embeddings + produit scalaire

Vecteurs normés empilés dans une matrice `(N, dimensions)` : le cosinus se réduit à
un `matmul`. À 10²–10³ documents la recherche exacte est instantanée — un moteur
type Elasticsearch/pgvector ne se justifie qu'au-delà, ou pour servir plusieurs
utilisateurs concurrents.

**Score d'une entité documentée** : `max(identité, mélange des deux canaux)`.
Pourquoi ce mélange précisément (calé sur le golden set de la section 8, pas choisi
a priori) :

| Fusion                     | recall@3 | Défaut                                                        |
| --------------------------- | -------- | --------------------------------------------------------------- |
| identité seule               | 2/5      | ignore toute description                                        |
| mélange seul                 | 3/5      | peut *baisser* une entité mal décrite mais bien nommée           |
| max des deux canaux bruts     | 3/5      | une phrase FR matche mieux une question FR qu'un identifiant     |
|                              |          | déplié → toute entité documentée domine, quelle que soit la question |
| **max(identité, mélange)**    | **4/5**  | monotone : documenter ne peut jamais nuire                       |

L'artefact (`embeddings.npy`, `meaning_embeddings.npy`, `documents.jsonl`,
`meta.json`) est persisté dans `INDEX_DIR`, avec une empreinte du corpus
(`corpus_hash`) pour détecter qu'il est périmé sans avoir à ré-embedder pour vérifier.

In [ ]:
def _embedder(model_name: str):
    """Import tardif : les cellules precedentes restent utilisables sans fastembed
    installe. Poids mis en cache dans MODEL_CACHE_DIR (le defaut de fastembed est
    /tmp, souvent un tmpfs -> retelechargement a chaque redemarrage).
    """
    from fastembed import TextEmbedding
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return TextEmbedding(model_name, cache_dir=str(MODEL_CACHE_DIR))


def _normalize(matrix: np.ndarray) -> np.ndarray:
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


def corpus_hash(documents: list[Document]) -> str:
    """Empreinte du corpus : detecte qu'un index sur disque est perime (une
    description ajoutee change le hash) sans avoir a reembedder pour verifier.
    """
    digest = hashlib.sha256()
    for d in documents:
        digest.update(d.fqn.encode("utf-8"))
        digest.update(d.identity_text.encode("utf-8"))
        digest.update(d.meaning_text.encode("utf-8"))
    return digest.hexdigest()[:16]


def _embed_meanings(model, documents: list[Document], dimensions: int) -> np.ndarray:
    """Embeddings du texte de sens ; ligne nulle pour les entites non documentees
    (on ne paie l'embedding que pour celles qui ont quelque chose a dire).
    """
    matrix = np.zeros((len(documents), dimensions))
    positions = [i for i, d in enumerate(documents) if d.has_meaning]
    if positions:
        vectors = _normalize(
            np.array(list(model.embed([documents[i].meaning_text for i in positions])))
        )
        matrix[positions] = vectors
    return matrix


@dataclass
class SearchHit:
    score: float
    document: Document


@dataclass
class SemanticIndex:
    """Deux matrices : les identifiants deplies et, quand elle existe, la
    documentation. `meaning_weight` et `model_name` sont geles a la construction :
    changer l'un des deux implique de reconstruire l'index (le vocabulaire de deux
    modeles differents ne vit jamais dans le meme espace vectoriel).
    """

    documents: list[Document]
    embeddings: np.ndarray  # canal identite
    meaning_embeddings: np.ndarray  # canal sens (lignes nulles si non documente)
    model_name: str = EMBEDDING_MODEL
    meaning_weight: float = MEANING_WEIGHT

    @classmethod
    def build(
        cls, documents: list[Document], model_name: str = EMBEDDING_MODEL,
        meaning_weight: float = MEANING_WEIGHT,
    ) -> "SemanticIndex":
        model = _embedder(model_name)
        embeddings = _normalize(np.array(list(model.embed([d.identity_text for d in documents]))))
        meanings = _embed_meanings(model, documents, embeddings.shape[1])
        log.info(
            "index construit : %s, dont %d document(s) documente(s)",
            embeddings.shape, sum(1 for d in documents if d.has_meaning),
        )
        return cls(
            documents=documents, embeddings=embeddings, meaning_embeddings=meanings,
            model_name=model_name, meaning_weight=meaning_weight,
        )

    def save(self, directory: Path = INDEX_DIR) -> Path:
        directory.mkdir(parents=True, exist_ok=True)
        np.save(directory / "embeddings.npy", self.embeddings)
        np.save(directory / "meaning_embeddings.npy", self.meaning_embeddings)
        with (directory / "documents.jsonl").open("w", encoding="utf-8") as fh:
            for d in self.documents:
                fh.write(json.dumps(d.to_dict(), ensure_ascii=False) + "\n")
        (directory / "meta.json").write_text(
            json.dumps({
                "model": self.model_name, "meaning_weight": self.meaning_weight,
                "documents": len(self.documents),
                "documented": sum(1 for d in self.documents if d.has_meaning),
                "dimensions": int(self.embeddings.shape[1]),
                "corpus_hash": corpus_hash(self.documents),
                "built_at": datetime.now(UTC).isoformat(),
            }, indent=2),
            encoding="utf-8",
        )
        return directory

    @classmethod
    def load(cls, directory: Path = INDEX_DIR) -> "SemanticIndex":
        meta = json.loads((directory / "meta.json").read_text(encoding="utf-8"))
        documents = [
            Document.from_dict(json.loads(line))
            for line in (directory / "documents.jsonl").read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        return cls(
            documents=documents,
            embeddings=np.load(directory / "embeddings.npy"),
            meaning_embeddings=np.load(directory / "meaning_embeddings.npy"),
            model_name=meta["model"], meaning_weight=meta.get("meaning_weight", MEANING_WEIGHT),
        )

    def is_stale(self, documents: list[Document]) -> bool:
        """True si le catalogue a bouge depuis la construction de l'index."""
        return corpus_hash(documents) != corpus_hash(self.documents)

    def search(
        self, question: str, top_k: int = 5, entity_type: str | None = None,
        schema: str | None = None, tags: set[str] | None = None,
        described_only: bool = False,
    ) -> list[SearchHit]:
        """Recherche filtree. Les filtres s'appliquent en masque sur les scores : a
        cette echelle, pre-filtrer le corpus n'apporte rien, et le masque garde la
        semantique simple ("les meilleurs parmi ceux qui passent le filtre").
        """
        model = _embedder(self.model_name)
        vector = next(iter(model.embed([question])))
        vector = vector / np.linalg.norm(vector)
        documented = np.array([d.has_meaning for d in self.documents])
        identity_scores = self.embeddings @ vector
        meaning_scores = self.meaning_embeddings @ vector
        blended = (1 - self.meaning_weight) * identity_scores + self.meaning_weight * meaning_scores
        scores = np.where(documented, np.maximum(identity_scores, blended), identity_scores)

        keep = np.ones(len(self.documents), dtype=bool)
        for i, d in enumerate(self.documents):
            if entity_type and d.entity_type != entity_type:
                keep[i] = False
            elif schema and d.schema.lower() != schema.lower():
                keep[i] = False
            elif tags and not tags.intersection(d.tags):
                keep[i] = False
            elif described_only and not d.has_description:
                keep[i] = False
        scores = np.where(keep, scores, -np.inf)

        best = np.argsort(-scores)[:top_k]
        return [
            SearchHit(score=float(scores[i]), document=self.documents[i])
            for i in best if np.isfinite(scores[i])
        ]

In [ ]:
start = time.perf_counter()
index = SemanticIndex.build(documents)
index.save()
documented = sum(1 for d in documents if d.has_meaning)
print(
    f"{index.embeddings.shape} en {time.perf_counter() - start:.1f}s "
    f"({documented} document(s) avec du sens) -> {INDEX_DIR}/"
)

In [ ]:
# Aller-retour disque : illustre l'artefact persiste et la detection de peremption.
reloaded = SemanticIndex.load()
print("recharge depuis disque :", reloaded.embeddings.shape)
print("perime par rapport au catalogue courant ?", reloaded.is_stale(documents))

## 6. Chercher

In [ ]:
def montre(question: str, index: SemanticIndex, k: int = 3, **filtres) -> None:
    print(f"\nQ: {question}")
    for hit in index.search(question, top_k=k, **filtres):
        doc = hit.document
        marque = " +desc" if doc.has_description else ""
        print(f"   {hit.score:.3f}  {doc.entity_type:6} {doc.name}{marque}")


QUESTIONS = [
    "solde journalier d'un compte",
    "mouvements debiteurs du mois",
    "capital restant du sur un credit",
    "adresse du client",
]
for question in QUESTIONS:
    montre(question, index, entity_type="table")

## 7. Filtrer par facette

Schéma, type d'entité, tags de classification, présence d'une description : autant
de masques combinables sur les scores.

In [ ]:
montre("solde d'un compte", index, entity_type="column", schema="dmt")
montre("solde d'un compte", index, entity_type="table", described_only=True)

## 8. Mesurer l'effet des métadonnées

Un « golden set » — cinq questions, la table attendue pour chacune — pour comparer
deux configurations autrement qu'à l'œil. Un vrai golden set fait plutôt ~50 entrées ;
cinq suffisent pour illustrer la mécanique.

Cette mesure porte toujours sur le **catalogue synthétique** (les réponses attendues
visent ses noms de table), indépendamment du `DATA_SOURCE` choisi en section 3.

In [ ]:
GOLDEN = {
    "solde journalier d'un compte": "DMT_CPT_SLD_J",
    "mouvements debiteurs du mois": "DMT_CPT_MVT_J",
    "capital restant du sur un credit": "DMT_F_CRD_ENC_M",
    "adresse du client": "ODS_D_CLI_ADR",
    "taux de change vers le franc suisse": "V_REF_FIN_TXC_CHF",
}


def recall_at_k(index: SemanticIndex, k: int = 3) -> None:
    trouves = []
    for question, attendu in GOLDEN.items():
        noms = [h.document.name for h in index.search(question, top_k=k, entity_type="table")]
        rang = noms.index(attendu) + 1 if attendu in noms else None
        trouves.append(rang is not None)
        etat = f"rang {rang}" if rang else "RATE"
        print(f"   {question:38} {etat:8} top1={noms[0]}")
    print(f"   recall@{k} = {sum(trouves)}/{len(GOLDEN)}\n")


index_brut = SemanticIndex.build(build_documents(load_synthetic_catalog(enriched=False)))
index_enrichi = SemanticIndex.build(build_documents(load_synthetic_catalog(enriched=True)))

print("-- catalogue brut (identifiants seuls)")
recall_at_k(index_brut)
print("-- catalogue enrichi (identifiants + descriptions + glossaire)")
recall_at_k(index_enrichi)

## 9. À retenir, et quoi tweaker

- **Échec connu** : `ODS_F_CPT_MVT_BCK_2019` (jamais documentée) remonte parfois sur
  des questions visant `DMT_CPT_MVT_J` — les tables `_BCK_*`/`_OLD` polluent le haut
  du classement. Un filtre par tag de classification (`schema`/`tags` dans `search`)
  est la bonne réponse ; encore faut-il que le tag existe dans le catalogue.
- **Régime asymétrique** : ici 5 tables sur 6 sont enrichies dans la variante
  `enriched=True` — sur un vrai catalogue, la proportion documentée sera bien plus
  faible au départ. `MEANING_WEIGHT` et le choix `max(identité, mélange)` sont calés
  sur ce golden set ; à rejouer quand la couverture de documentation change beaucoup.

**Ce qui est fait pour être changé (cellule de configuration, section 0) :**

| Variable                 | Effet                                                             |
| ------------------------ | -------------------------------------------------------------------- |
| `DATA_SOURCE`            | `"synthetic"` (démo hors-ligne) ↔ `"omd"` (catalogue réel)            |
| `EMBEDDING_MODEL`        | tout modèle fastembed — comparer un modèle anglais sur du texte FR   |
| `MEANING_WEIGHT`         | 0 = ignore les descriptions, 1 = ignore les noms de colonnes         |
| `INCLUDE_COLUMNS`        | `False` pour indexer les tables seules (plus rapide, moins fin)      |
| `INDEX_DIR`              | où persister l'artefact (un répertoire par expérience, par exemple) |
| `GOLDEN`                 | ajouter ses propres questions/réponses pour mesurer sur son domaine  |
| `load_synthetic_catalog` | ajouter ses propres tables pour un autre domaine métier              |